In [0]:
%run "./00_Pipeline_Configurations"

In [0]:
%run "./08_Audit_Log_Function"

In [0]:
from datetime import datetime
from pyspark.sql.functions import col, round, when

start_time = datetime.now()

try:
    df_attendance_gold = (
        spark.table(silver_attendance_table)
        .withColumn("total_marks", col("math_marks") + col("science_marks"))
        .withColumn("avg_marks", round((col("math_marks") + col("science_marks")) / 2, 2))
        .withColumn("result_status", when(col("avg_marks") >= 40, "Pass").otherwise("Fail"))
    )

    df_attendance_gold.write.format("delta").mode("overwrite").saveAsTable(gold_attendance_table)

    display(df_attendance_gold)

    print(f"Total rows in gold_attendance: {df_attendance_gold.count()}")

    log_audit("gold_attendance", table_name=gold_attendance_table, status="SUCCESS", start_time=start_time)

except Exception as e:
    log_audit("gold_attendance", status="FAILED", start_time=start_time, error_msg=str(e))
    raise